In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re

In [2]:
file_path = "../data/preprocessed/preprocessed_en-2020-pssd-compendium.csv"
df = pd.read_csv(file_path)

In [3]:
df = df[["Job title", "Salary"]].dropna()   # simplifying, only keeping job title and salary for now
df = df.sample(frac=1/50, random_state=42)  # sampling 1/50th of the data for now

In [4]:
# cleaning salary (maybe we should do this in preprocessing)
df["Salary"] = (
    df["Salary"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

In [5]:
def tokenize(title):
    title = re.sub(r"[^a-zA-Z0-9, ]+", "", title.lower())
    return title.split()

In [6]:
vocab = sorted(list({w for title in df["Job title"] for w in tokenize(title)}))
stoi = {ch: i + 1 for i, ch in enumerate(vocab)}    # string to int, use 0s for padding
itos = {i: ch for ch, i in stoi.items()}    # int to string (inverse)

In [7]:
def encode(title, max_len=64):
    ids = [stoi.get(c, 0) for c in title.lower()[:max_len]]
    return ids + [0] * (max_len - len(ids))     # padding

In [8]:
class SalaryDataset(Dataset):
    def __init__(self, df):
        self.x = torch.tensor([encode(t) for t in df["Job title"]])
        self.y = torch.tensor(df["Salary"].values, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

In [9]:
dataset = SalaryDataset(df)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [10]:
class SalaryTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, d_model)  # 0 used for padding so vocab_size + 1
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.regressor = nn.Linear(d_model, 1)  # regression
    def forward(self, x):
        emb = self.embedding(x).transpose(0, 1)  # seq_len x batch x d_model
        encoded = self.transformer(emb)
        pooled = encoded.mean(dim=0)  # batch x d_model
        return self.regressor(pooled).squeeze(-1)

In [11]:
model = SalaryTransformer(len(vocab))
criterion = nn.MSELoss()    # mean squared error loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)     # using Adam can change to whatever

C:\Users\Daniyaal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [12]:
for epoch in range(200):
    for x, y in loader:
        pred = model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, loss={loss.item():.2f}")

Epoch 0, loss=9866991616.00
Epoch 1, loss=9449976832.00
Epoch 2, loss=8843287552.00
Epoch 3, loss=9307514880.00
Epoch 4, loss=11499676672.00
Epoch 5, loss=6236717568.00
Epoch 6, loss=5065333248.00
Epoch 7, loss=7412257792.00
Epoch 8, loss=3200259584.00
Epoch 9, loss=4401093120.00
Epoch 10, loss=1717941248.00
Epoch 11, loss=1172927104.00
Epoch 12, loss=43256620.00
Epoch 13, loss=204508768.00
Epoch 14, loss=18213890.00
Epoch 15, loss=218021600.00
Epoch 16, loss=223819360.00
Epoch 17, loss=1075887.50
Epoch 18, loss=2418227456.00
Epoch 19, loss=424680960.00
Epoch 20, loss=521327328.00
Epoch 21, loss=33340956672.00
Epoch 22, loss=1231719936.00
Epoch 23, loss=442520192.00
Epoch 24, loss=647140352.00
Epoch 25, loss=46373228.00
Epoch 26, loss=260963584.00
Epoch 27, loss=52823712.00
Epoch 28, loss=2941792000.00
Epoch 29, loss=21466716.00
Epoch 30, loss=500884672.00
Epoch 31, loss=107047472.00
Epoch 32, loss=312237184.00
Epoch 33, loss=19788804.00
Epoch 34, loss=71121024.00
Epoch 35, loss=661589

In [13]:
with torch.no_grad():
    test_title = "Chair"
    test_input = torch.tensor([encode(test_title)])
    pred_salary = model(test_input).item()
    print(f"Predicted salary for '{test_title}': ${pred_salary:.2f}")

Predicted salary for 'Chair': $117876.45
